<a href="https://colab.research.google.com/github/adillach/Hackativ8/blob/master/Salinan_dari_AI_Agents_AVPN_IT_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agents with LangGraph Part 1

In [34]:
%pip install -qU langgraph langchain-google-genai

## Exercise 1: SQL Agents

SQL agents adalah AI Agent yang dapat mengambil data dari table SQL dan menganalisis sesuai dengan pertanyaan dari user.

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

from google.colab import userdata
GEMINI = userdata.get('GEMINI')
os.environ["GOOGLE_API_KEY"] = GEMINI

### Langkah 1: Define beberapa table SQL

In [36]:
%load_ext sql
%sql sqlite:///sample.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [37]:
%%sql
-- Create the 'products' table
CREATE TABLE IF NOT EXISTS products (
  	product_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	product_name VARCHAR(255) NOT NULL,
  	price DECIMAL(10, 2) NOT NULL
  );

-- Create the 'staff' table
CREATE TABLE IF NOT EXISTS staff (
  	staff_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	first_name VARCHAR(255) NOT NULL,
  	last_name VARCHAR(255) NOT NULL
  );

-- Create the 'orders' table
CREATE TABLE IF NOT EXISTS orders (
  	order_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	customer_name VARCHAR(255) NOT NULL,
  	staff_id INTEGER NOT NULL,
  	product_id INTEGER NOT NULL,
  	FOREIGN KEY (staff_id) REFERENCES staff (staff_id),
  	FOREIGN KEY (product_id) REFERENCES products (product_id)
  );

-- Insert data into the 'products' table
INSERT INTO products (product_name, price) VALUES
  	('Laptop', 799.99),
  	('Keyboard', 129.99),
  	('Mouse', 29.99);

-- Insert data into the 'staff' table
INSERT INTO staff (first_name, last_name) VALUES
  	('Alice', 'Smith'),
  	('Bob', 'Johnson'),
  	('Charlie', 'Williams');

-- Insert data into the 'orders' table
INSERT INTO orders (customer_name, staff_id, product_id) VALUES
  	('David Lee', 1, 1),
  	('Emily Chen', 2, 2),
  	('Frank Brown', 1, 3);

 * sqlite:///sample.db
Done.
Done.
Done.
3 rows affected.
3 rows affected.
3 rows affected.


[]

In [38]:
import sqlite3

db_file = "sample.db"

### Langkah 2: Define Tools
Tools yang akan digunakan pada SQL agent adalah `list_tables`, `describe_tables`, dan `execute_query` sehingga AI dapat memahami metadata table terlebih dahulu sebelum mulai menjalankan query

In [39]:
def list_tables() -> list[str]:
    """Retrieve the names of all tables in the database."""
    # Include print logging statements so you can see when functions are being called.
    print(' - DB CALL: list_tables')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        # Fetch the table names.
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

        tables = cursor.fetchall()
        return [t[0] for t in tables]


list_tables()

 - DB CALL: list_tables


['products', 'sqlite_sequence', 'staff', 'orders']

In [40]:
def describe_table(table_name: str) -> list[tuple[str, str]]:
    """Look up the table schema.

    Returns:
      List of columns, where each entry is a tuple of (column, type).
    """
    print(' - DB CALL: describe_table')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        cursor.execute(f"PRAGMA table_info({table_name});")

        schema = cursor.fetchall()
        # [column index, column name, column type, ...]
        return [(col[1], col[2]) for col in schema]


describe_table("products")

 - DB CALL: describe_table


[('product_id', 'INTEGER'),
 ('product_name', 'VARCHAR(255)'),
 ('price', 'DECIMAL(10, 2)')]

In [41]:
def execute_query(sql: str) -> list[list[str]]:
    """Execute a SELECT statement, returning the results."""
    print(' - DB CALL: execute_query')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        cursor.execute(sql)
        return cursor.fetchall()


execute_query("select * from products")

 - DB CALL: execute_query


[(1, 'Laptop', 799.99),
 (2, 'Keyboard', 129.99),
 (3, 'Mouse', 29.99),
 (4, 'Laptop', 799.99),
 (5, 'Keyboard', 129.99),
 (6, 'Mouse', 29.99)]

### Langkah 3: Bangun chatbot dengan create_react_agent

In [42]:
# These are the Python functions defined above.
db_tools = [list_tables, describe_table, execute_query]

instruction = """You are a helpful chatbot that can interact with an SQL database for a computer
store. You will take the users questions and turn them into SQL queries using the tools
available. Once you have the information you need, you will answer the user's question using
the data returned. ALWAYS start by calling list_tables to discover the available tables.
ALWAYS call describe_table on the relevant table before writing any SQL query.
Never assume or guess table names or column names — always verify first."""

# Import the necessary modules from langgraph
from langchain.agents import create_agent

# Initialize the chat model with specific parameters
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Create a ReAct agent using the model, tools, and prompt
agent = create_agent(
    model=model,
    tools=db_tools,
    system_prompt=instruction
)

perhatikan respons chatbot di bawah, LLM memanggil tool list_table -> describe_table -> execute_query untuk menganalisis jawaban dan memberikan jawaban yang tepat.

In [44]:
# Run the agent
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Siapa saja orang yang terdaftar di database ini?"}]}
)

print(response["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 40.449280424s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM

In [11]:
# Run the agent with a different query
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Barang apa yang dijual di toko ini dan berapa harganya?"}]}
)

print(response["messages"][-1].content)

 - DB CALL: list_tables
 - DB CALL: describe_table
Toko ini menjual produk-produk berikut beserta harganya:

*   **Product ID**: 1, **Product Name**: Laptop, **Price**: 1200.00
*   **Product ID**: 2, **Product Name**: Keyboard, **Price**: 75.00
*   **Product ID**: 3, **Product Name**: Mouse, **Price**: 25.00
*   **Product ID**: 4, **Product Name**: Monitor, **Price**: 300.00
*   **Product ID**: 5, **Product Name**: Webcam, **Price**: 50.00
*   **Product ID**: 6, **Product Name**: Microphone, **Price**: 100.00
*   **Product ID**: 7, **Product Name**: Printer, **Price**: 250.00
*   **Product ID**: 8, **Product Name**: USB Cable, **Price**: 10.00
*   **Product ID**: 9, **Product Name**: External Hard Drive, **Price**: 150.00
*   **Product ID**: 10, **Product Name**: Desk Lamp, **Price**: 30.00


# AI Agents with LangGraph Part 2

## Langkah 1: Define State dan Prompt

In [22]:
system_instruction = """
You are a Customer Service Bot for an online bookstore. A human will talk to you about available books and you will answer any questions about them.
The customer will add 1 or more books to their shopping cart, which you will manage and send to a purchase system after confirming the order with the human.

Add items to the customer's cart with `add_to_cart`, and reset the cart with `clear_cart`.
To see the contents of the cart so far, call `get_cart` (this is shown to you, not the user).
Always `confirm_purchase` with the user (double-check) before calling `make_purchase`.
Calling `confirm_purchase` will display the cart items to the user and returns their response to seeing the list.
Their response may contain modifications. Always verify and respond with book titles and ISBNs from the AVAILABLE BOOKS list before adding them to the cart.
If you are unsure a book matches those on the list, ask a question to clarify or redirect.

Once the customer has finished adding items, call `confirm_purchase` to ensure it is correct, then make any necessary updates and then call `make_purchase`.
Once `make_purchase` has returned, thank the user and say goodbye!
"""

## Langkah 2: Define Flow Chatbot dengan create_react_agent

In [23]:
# In-memory database for the customer's cart
customer_cart = []

def get_book_list() -> str:
    """Provide the latest up-to-date list of available books."""

    return """
    AVAILABLE BOOKS:
    Fiction:
    "The Great Gatsby" by F. Scott Fitzgerald (ISBN: 978-0743273565, Price: $12.99)
    "1984" by George Orwell (ISBN: 978-0451524935, Price: $9.99)
    "To Kill a Mockingbird" by Harper Lee (ISBN: 978-0061120084, Price: $10.50)

    Non-Fiction:
    "Sapiens: A Brief History of Humankind" by Yuval Noah Harari (ISBN: 978-0062316097, Price: $16.00)
    "Educated" by Tara Westover (ISBN: 978-0399590504, Price: $14.99)

    Children's Books:
    "The Very Hungry Caterpillar" by Eric Carle (ISBN: 978-0399208539, Price: $7.99)
    "Where the Wild Things Are" by Maurice Sendak (ISBN: 978-0060254926, Price: $8.50)

    Special Offers:
    "Mystery Thriller Bundle" (Includes 3 random mystery novels, Price: $25.00)
    "Classic Literature Collection" (Includes 5 classic novels, Price: $40.00)
    """

def add_to_cart(item: str) -> str:
    """Add an item to the customer's shopping cart."""
    global customer_cart
    customer_cart.append(item)
    print(f"Adding '{item}' to cart. Current cart: {customer_cart}")
    return f"I've added '{item}' to your cart."

def clear_cart() -> str:
    """Clear all items from the customer's shopping cart."""
    global customer_cart
    customer_cart.clear()
    print("Clearing cart.")
    return "Your cart has been cleared."

def get_cart() -> list[str]:
    """Get the current items in the customer's shopping cart."""
    global customer_cart
    print(f"Getting cart. Current cart: {customer_cart}")
    return customer_cart

def confirm_purchase() -> str:
    """Confirm the shopping cart contents with the customer."""
    global customer_cart
    print(f"Confirming cart. Current cart: {customer_cart}")
    if not customer_cart:
        return "Your cart is currently empty. What books can I help you with?"

    cart_string = ", ".join(customer_cart)
    return f"Your cart contains: {cart_string}. Is this correct, or would you like to make any changes?"

def make_purchase() -> str:
    """Finalize the purchase."""
    global customer_cart
    print(f"Making purchase. Final cart: {customer_cart}")
    if not customer_cart:
        return "There's nothing in your cart to purchase."

    final_purchase_summary = ", ".join(customer_cart)
    # Clear the cart after purchase, ready for the next customer
    customer_cart.clear()
    return f"Your purchase for '{final_purchase_summary}' has been processed! Thank you for shopping with us!"

# Collect all tools
bookstore_tools = [get_book_list, add_to_cart, clear_cart, get_cart, confirm_purchase, make_purchase]

# Import necessary modules
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Initialize the chat model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.2
)

# Create a checkpointer for memory
checkpointer = InMemorySaver()

# Create the ReAct agent
bookstore_agent = create_agent(
    model=model,
    tools=bookstore_tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)


### Interaksi dengan Barista Bot

In [24]:
# Function to interact with the bookstore bot
def chat_with_bookstore_bot():
    thread_id = "1"
    config = {"configurable": {"thread_id": thread_id}}

    print("Welcome to our online bookstore! Type 'q' to quit.")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ['q', 'quit', 'exit']:
            print("Bookstore Bot: Thank you for visiting! Have a great day!")
            break

        response = bookstore_agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": user_input
                    }
                ]
            },
            config=config
        )

        print("Bookstore Bot:", response["messages"][-1].content)


In [29]:
# Run the interactive chat
# Uncomment the line below to start chatting with the bookstore bot
chat_with_bookstore_bot()

Welcome to our online bookstore! Type 'q' to quit.
You: q
Bookstore Bot: Thank you for visiting! Have a great day!


### Contoh Penggunaan Langsung

In [28]:
# Example of direct usage with specific queries
thread_id = "3"  # Unique identifier for this conversation
config = {"configurable": {"thread_id": thread_id}}

# User asks about available books
response = bookstore_agent.invoke(
    {"messages": [{"role": "user", "content": "What books do you have?"}]},
    config=config
)
print("User: What books do you have?")
print("Bookstore Bot:", response["messages"][-1].content)


User: What books do you have?
Bookstore Bot: 


In [33]:
# User adds a book to cart
response = bookstore_agent.invoke(
    {"messages": [{"role": "user", "content": "I'd like to add \"1984\" to my cart please"}]},
    config=config
)
print("User: I'd like to add \"1984\" to my cart please")
print("Bookstore Bot:", response["messages"][-1].content)


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 42.262992689s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM

In [ ]:
# User checks their cart
response = bookstore_agent.invoke(
    {"messages": [{"role": "user", "content": "what is in my cart?"}]},
    config=config
)
print("User: what is in my cart?")
print("Bookstore Bot:", response["messages"][-1].content)
